# Demo — replica-inpc-mx v2

Demuestra el uso del proyecto con **datos sintéticos**.
Los valores de índice y ponderadores son ficticios y no representan datos oficiales del INEGI.

Para datos reales, ver `docs/uso.md`.

In [ ]:
import replica_inpc as rep

CANASTA = "canasta_demo.csv"
SERIES  = "series_demo.csv"

# Sin token: el cálculo corre igual, la validación queda como no_disponible.
# rep.set_token("tu-token-inegi")

## 1. Cargar insumos

In [ ]:
can = rep.cargar_canasta(CANASTA, 2018)
ser = rep.cargar_serie(SERIES, 2018)

print(f"Canasta: {len(can.df)} genéricos")
print(f"Serie:   {len(ser.df.columns)} periodos ({ser.df.columns[0]} → {ser.df.columns[-1]})")

## 2. Calcular INPC

In [ ]:
inpc = rep.calcular_indice(can, ser, "inpc")
inpc.resumen

In [ ]:
# Resultado en formato largo (MultiIndex periodo x indice)
inpc.resultado.largo

In [ ]:
# Resultado en formato ancho (periodos como columnas)
inpc.resultado.ancho

## 3. Rebasar a base diferente

In [ ]:
# Los datos demo parten de 2Q Jul 2018 = 100
inpc_rebased = rep.rebasar(inpc, "2Q Jul 2018")
inpc_rebased.resultado.ancho

## 4. Variación periódica (inflación quincenal)

In [ ]:
var = rep.variacion_periodica(inpc, frecuencia="quincenal")
var.resumen

In [ ]:
var.resultado.ancho

## 5. Consultas de inflación

In [ ]:
# inflacion_en: DataFrame con todos los índices en ese periodo
rep.inflacion_en(var, "1Q Ene 2019")

In [ ]:
print("Inflación acumulada 1Q Ago 2018 → 1Q Ene 2019:",
      rep.inflacion_acumulada(var, "1Q Ago 2018", "1Q Ene 2019", indice="INPC"))

periodo, indice, valor = rep.inflacion_maxima(var, indice="INPC")
print(f"Inflación máxima INPC: {valor:.4f}% en {periodo}")

## 6. Subíndices por componente de inflación

In [ ]:
comp = rep.calcular_indice(can, ser, "inflacion componente")
comp.resumen

In [ ]:
comp.resultado.ancho

## 7. Validación contra INEGI (requiere token)

Sin token, `estado_validacion` queda en `no_disponible`.

In [ ]:
from replica_inpc.dominio.errores import ErrorConfiguracion

try:
    val = rep.validar_indice(inpc)
    display(val.resumen)
except ErrorConfiguracion as e:
    print(f"Sin token INEGI: {e}")

In [ ]:
try:
    display(val.resultado.ancho)
except NameError:
    pass